In [1]:
import os
import re
import math
import warnings
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm
from peft import PeftModel

warnings.filterwarnings("ignore")

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


CUDA available: True
GPU: Tesla P100-PCIE-16GB


In [2]:
!pip install /kaggle/input/datasets/jiayuanshe/package/evaluate-0.4.6-py3-none-any.whl
!pip install /kaggle/input/datasets/jiayuanshe/package/portalocker-3.2.0-py3-none-any.whl
!pip install /kaggle/input/datasets/jiayuanshe/package/sacrebleu-2.6.0-py3-none-any.whl

Processing /kaggle/input/datasets/jiayuanshe/package/evaluate-0.4.6-py3-none-any.whl
Processing /kaggle/input/datasets/jiayuanshe/package/portalocker-3.2.0-py3-none-any.whl
Processing /kaggle/input/datasets/jiayuanshe/package/sacrebleu-2.6.0-py3-none-any.whl


In [3]:
import sacrebleu

In [4]:
## ── Preprocessing helpers ──────────────────────────────────────────────────

_V2 = re.compile(r"([aAeEiIuU])(?:2|₂)")
_V3 = re.compile(r"([aAeEiIuU])(?:3|₃)")
_ACUTE = str.maketrans({"a":"á","e":"é","i":"í","u":"ú","A":"Á","E":"É","I":"Í","U":"Ú"})
_GRAVE = str.maketrans({"a":"à","e":"è","i":"ì","u":"ù","A":"À","E":"È","I":"Ì","U":"Ù"})

def _ascii_to_diacritics(s: str) -> str:
    s = s.replace("sz", "š").replace("SZ", "Š")
    s = s.replace("s,", "ṣ").replace("S,", "Ṣ")
    s = s.replace("t,", "ṭ").replace("T,", "Ṭ")
    s = _V2.sub(lambda m: m.group(1).translate(_ACUTE), s)
    s = _V3.sub(lambda m: m.group(1).translate(_GRAVE), s)
    return s

_ALLOWED_FRACS = [
    (1/6, "0.16666"), (1/4, "0.25"), (1/3, "0.33333"),
    (1/2, "0.5"), (2/3, "0.66666"), (3/4, "0.75"), (5/6, "0.83333"),
]
_FRAC_TOL = 2e-3
_FLOAT_RE = re.compile(r"(?<![\w/])(\d+\.\d{4,})(?![\w/])")

def _canon_decimal(x: float) -> str:
    ip = int(math.floor(x + 1e-12))
    frac = x - ip
    best = min(_ALLOWED_FRACS, key=lambda t: abs(frac - t[0]))
    if abs(frac - best[0]) <= _FRAC_TOL:
        dec = best[1]
        if ip == 0:
            return dec
        return f"{ip}{dec[1:]}" if dec.startswith("0.") else f"{ip}+{dec}"
    return f"{x:.5f}".rstrip("0").rstrip(".")

_WS_RE = re.compile(r"\s+")

_GAP_UNIFIED_RE = re.compile(
    r"<\s*big[\s_\-]*gap\s*>"
    r"|<\s*gap\s*>"
    r"|\bbig[\s_\-]*gap\b"
    r"|\bx(?:\s+x)+\b"
    r"|\.{3,}|…+|\[\.+\]"
    r"|\[\s*x\s*\]|\(\s*x\s*\)"
    r"|(?<!\w)x{2,}(?!\w)"
    r"|(?<!\w)x(?!\w)"
    r"|\(\s*large\s+break\s*\)"
    r"|\(\s*break\s*\)"
    r"|\(\s*\d+\s+broken\s+lines?\s*\)",
    re.I
)

def _normalize_gaps_vec(ser: pd.Series) -> pd.Series:
    return ser.str.replace(_GAP_UNIFIED_RE, "<gap>", regex=True)

_CHAR_TRANS = str.maketrans({
    "ḫ":"h","Ḫ":"H","ʾ":"",
    "₀":"0","₁":"1","₂":"2","₃":"3","₄":"4",
    "₅":"5","₆":"6","₇":"7","₈":"8","₉":"9",
    "—":"-","–":"-",
})
_SUB_X = "ₓ"

_UNICODE_UPPER = r"A-ZŠṬṢḪ\u00C0-\u00D6\u00D8-\u00DE\u0160\u1E00-\u1EFF"
_UNICODE_LOWER = r"a-zšṭṣḫ\u00E0-\u00F6\u00F8-\u00FF\u0161\u1E01-\u1EFF"
_DET_UPPER_RE = re.compile(r"\(([" + _UNICODE_UPPER + r"0-9]{1,6})\)")
_DET_LOWER_RE = re.compile(r"\(([" + _UNICODE_LOWER + r"]{1,4})\)")

_PN_RE = re.compile(r"\bPN\b")
_KUBABBAR_RE = re.compile(r"KÙ\.B\.")

_EXACT_FRAC_RE = re.compile(r"0\.8333|0\.6666|0\.3333|0\.1666|0\.625|0\.75|0\.25|0\.5")
_EXACT_FRAC_MAP = {
    "0.8333": "⅚", "0.6666": "⅔", "0.3333": "⅓", "0.1666": "⅙",
    "0.625": "⅝", "0.75": "¾", "0.25": "¼", "0.5": "½",
}

def _frac_repl(m: re.Match) -> str:
    return _EXACT_FRAC_MAP[m.group(0)]


class OptimizedPreprocessor:
    """Normalize Akkadian transliteration before tokenization."""

    def preprocess_batch(self, texts):
        ser = pd.Series(texts).fillna("").astype(str)
        ser = ser.apply(_ascii_to_diacritics)
        ser = ser.str.replace(_DET_UPPER_RE, r"\1", regex=True)
        ser = ser.str.replace(_DET_LOWER_RE, r"{\1}", regex=True)
        ser = _normalize_gaps_vec(ser)
        ser = ser.str.translate(_CHAR_TRANS)
        ser = ser.str.replace(_SUB_X, "", regex=False)
        ser = ser.str.replace(_KUBABBAR_RE, "KÙ.BABBAR", regex=True)
        ser = ser.str.replace(_EXACT_FRAC_RE, _frac_repl, regex=True)
        ser = ser.str.replace(_FLOAT_RE, lambda m: _canon_decimal(float(m.group(1))), regex=True)
        ser = ser.str.replace(_WS_RE, " ", regex=True).str.strip()
        return ser.tolist()


## ── Postprocessing helpers ─────────────────────────────────────────────────

_SOFT_GRAM_RE = re.compile(
    r"\(\s*(?:fem|plur|pl|sing|singular|plural|\?|\!)"
    r"(?:\.\s*(?:plur|plural|sing|singular))?"
    r"\.?\s*[^)]*\)", re.I
)
_BARE_GRAM_RE = re.compile(r"(?<!\w)(?:fem|sing|pl|plural)\.?(?!\w)\s*", re.I)
_UNCERTAIN_RE = re.compile(r"\(\?\)")
_CURLY_DQ_RE = re.compile("[\u201c\u201d]")   # " " → "
_CURLY_SQ_RE = re.compile("[\u2018\u2019]")   # ' ' → '
_MONTH_RE = re.compile(r"\bMonth\s+(XII|XI|X|IX|VIII|VII|VI|V|IV|III|II|I)\b", re.I)
_ROMAN2INT = {"I":1,"II":2,"III":3,"IV":4,"V":5,"VI":6,"VII":7,"VIII":8,"IX":9,"X":10,"XI":11,"XII":12}
_REPEAT_WORD_RE = re.compile(r"\b(\w+)(?:\s+\1\b)+")
_REPEAT_PUNCT_RE = re.compile(r"([.,])\1+")
_PUNCT_SPACE_RE = re.compile(r"\s+([.,:])") 
_FORBIDDEN_TRANS = str.maketrans("", "", '——<>⌈⌋⌊[]+ʾ;')
_COMMODITY_RE = re.compile(r'(?<=\s)-(gold|tax|textiles)\b')
_COMMODITY_REPL = {"gold": "pašallum gold", "tax": "šadduātum tax", "textiles": "kutānum textiles"}

def _commodity_repl(m: re.Match) -> str:
    return _COMMODITY_REPL[m.group(1)]

_SHEKEL_REPLS = [
    (re.compile(r'5\s+11\s*/\s*12\s+shekels?', re.I), '6 shekels less 15 grains'),
    (re.compile(r'5\s*/\s*12\s+shekels?', re.I), '⅓ shekel 15 grains'),
    (re.compile(r'7\s*/\s*12\s+shekels?', re.I), '½ shekel 15 grains'),
    (re.compile(r'1\s*/\s*12\s*(?:\(shekel\)|\bshekel)?', re.I), '15 grains'),
]
_SLASH_ALT_RE = re.compile(r'(?<![0-9/])\s+/\s+(?![0-9])\S+')
_STRAY_MARKS_RE = re.compile(r'<<[^>]*>>|<(?!gap\b)[^>]*>')
_MULTI_GAP_RE = re.compile(r'(?:<gap>\s*){2,}')
_EXTRA_STRAY_RE = re.compile(r'(?<!\w)(?:\.\.+|xx+)(?!\w)')
_HACEK_TRANS = str.maketrans({"ḫ": "h", "Ḫ": "H"})

def _month_repl(m: re.Match) -> str:
    return f"Month {_ROMAN2INT.get(m.group(1).upper(), m.group(1))}"


class VectorizedPostprocessor:
    """Clean and normalize model output translations."""

    def postprocess_batch(self, translations):
        s = pd.Series(translations).fillna("").astype(str)
        s = _normalize_gaps_vec(s)
        s = s.str.replace(_PN_RE, "<gap>", regex=True)
        s = s.str.replace(_COMMODITY_RE, _commodity_repl, regex=True)
        for pat, repl in _SHEKEL_REPLS:
            s = s.str.replace(pat, repl, regex=True)
        s = s.str.replace(_EXACT_FRAC_RE, _frac_repl, regex=True)
        s = s.str.replace(_FLOAT_RE, lambda m: _canon_decimal(float(m.group(1))), regex=True)
        s = s.str.replace(_SOFT_GRAM_RE, " ", regex=True)
        s = s.str.replace(_BARE_GRAM_RE, " ", regex=True)
        s = s.str.replace(_UNCERTAIN_RE, "", regex=True)
        s = s.str.replace(_STRAY_MARKS_RE, "", regex=True)
        s = s.str.replace(_EXTRA_STRAY_RE, "", regex=True)
        s = s.str.replace(_SLASH_ALT_RE, "", regex=True)
        s = s.str.replace(_CURLY_DQ_RE, '"', regex=True)
        s = s.str.replace(_CURLY_SQ_RE, "'", regex=True)
        s = s.str.replace(_MONTH_RE, _month_repl, regex=True)
        s = s.str.replace(_MULTI_GAP_RE, "<gap>", regex=True)
        # Protect <gap> markers before stripping forbidden chars
        s = s.str.replace("<gap>", "\x00GAP\x00", regex=False)
        s = s.str.translate(_FORBIDDEN_TRANS)
        s = s.str.replace("\x00GAP\x00", " <gap> ", regex=False)
        s = s.str.translate(_HACEK_TRANS)
        s = s.str.replace(_REPEAT_WORD_RE, r"\1", regex=True)
        for n in range(4, 1, -1):
            pat = r"\b((?:\w+\s+){" + str(n-1) + r"}\w+)(?:\s+\1\b)+"
            s = s.str.replace(pat, r"\1", regex=True)
        s = s.str.replace(_PUNCT_SPACE_RE, r"\1", regex=True)
        s = s.str.replace(_REPEAT_PUNCT_RE, r"\1", regex=True)
        s = s.str.replace(_WS_RE, " ", regex=True).str.strip()
        return s.tolist()


print("OptimizedPreprocessor and VectorizedPostprocessor loaded.")


OptimizedPreprocessor and VectorizedPostprocessor loaded.


In [5]:
## ── MBR selector (from regex notebook, single-model version) ───────────────

class MBRSelector:
    def __init__(
        self,
        pool_cap: int = 32,
        w_chrf: float = 0.55,
        w_bleu: float = 0.25,
        w_jaccard: float = 0.20,
        w_length: float = 0.10,
    ):
        self._chrf_metric = sacrebleu.metrics.CHRF(word_order=2)
        self._bleu_metric = sacrebleu.metrics.BLEU(effective_order=True)
        self.pool_cap = pool_cap
        self.w_chrf = w_chrf
        self.w_bleu = w_bleu
        self.w_jaccard = w_jaccard
        self.w_length = w_length
        self._pw_total = max(w_chrf + w_bleu + w_jaccard, 1e-9)

    def _chrfpp(self, a: str, b: str) -> float:
        if not a or not b:
            return 0.0
        return float(self._chrf_metric.sentence_score(a, [b]).score)

    def _bleu(self, a: str, b: str) -> float:
        if not a or not b:
            return 0.0
        try:
            return float(self._bleu_metric.sentence_score(a, [b]).score)
        except Exception:
            return 0.0

    @staticmethod
    def _jaccard(a: str, b: str) -> float:
        ta, tb = set(a.lower().split()), set(b.lower().split())
        if not ta and not tb:
            return 100.0
        if not ta or not tb:
            return 0.0
        return 100.0 * len(ta & tb) / len(ta | tb)

    def _pairwise_score(self, a: str, b: str) -> float:
        s = (
            self.w_chrf * self._chrfpp(a, b)
            + self.w_bleu * self._bleu(a, b)
            + self.w_jaccard * self._jaccard(a, b)
        )
        return s / self._pw_total

    @staticmethod
    def _length_bonus(lengths, idx: int) -> float:
        if len(lengths) == 0:
            return 100.0
        median = float(np.median(lengths))
        sigma = max(median * 0.4, 5.0)
        z = (lengths[idx] - median) / sigma
        return 100.0 * math.exp(-0.5 * z * z)

    @staticmethod
    def _dedup(xs):
        seen, out = set(), []
        for x in xs:
            x = str(x).strip()
            if x and x not in seen:
                out.append(x)
                seen.add(x)
        return out

    def pick(self, candidates):
        cands = self._dedup(candidates)
        if self.pool_cap:
            cands = cands[:self.pool_cap]

        n = len(cands)
        if n == 0:
            return ""
        if n == 1:
            return cands[0]

        lengths = [len(c.split()) for c in cands]
        scores = []

        for i in range(n):
            pw = sum(
                self._pairwise_score(cands[i], cands[j])
                for j in range(n) if j != i
            ) / max(1, n - 1)

            lb = self._length_bonus(lengths, i)
            total = pw + self.w_length * lb
            scores.append(total)

        return cands[int(np.argmax(scores))]

print("MBRSelector loaded.")

MBRSelector loaded.


## 1. Load Trained Model

In [ ]:
ADAPTER_DIR = "/kaggle/input/models/jiayuanshe/akka2eng-lora-02/transformers/default/1" 
BASE_MODEL_DIR = "/kaggle/input/models/jiayuanshe/byt5-large-base/transformers/default/1"

assert os.path.exists(ADAPTER_DIR), f"Adapter dir not found: {ADAPTER_DIR}"
assert os.path.exists(BASE_MODEL_DIR), f"Base model dir not found: {BASE_MODEL_DIR}"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_DIR, local_files_only=True)

base_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL_DIR,
    local_files_only=True
)

# load base model + adapter
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
    local_files_only=True
)

# model = model.merge_and_unload()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

print(f"Adapter loaded from: {ADAPTER_DIR}")
print(f"Base model loaded from: {BASE_MODEL_DIR}")
print(f"Model type: {type(model).__name__} (LoRA)")
print(f"Running on device: {device}")

Loading weights:   0%|          | 0/498 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Adapter loaded from: /kaggle/input/models/jiayuanshe/akka2eng-lora-02/transformers/default/1
Base model loaded from: /kaggle/input/models/jiayuanshe/byt5-large-base/transformers/default/1
Model type: PeftModelForSeq2SeqLM (LoRA)
Running on device: cuda


## 2. Load Test Data

In [7]:
# Load test data
TEST_DATA_DIR = "/kaggle/input/competitions/deep-past-initiative-machine-translation"
test_csv_path = os.path.join(TEST_DATA_DIR, "test.csv")

assert os.path.exists(test_csv_path), f"Test data not found: {test_csv_path}"

test_df = pd.read_csv(test_csv_path)

print(f"Test data shape: {test_df.shape}")
print(f"Test columns: {list(test_df.columns)}")
print(f"Sample test data:")
print(test_df.head())

Test data shape: (4, 5)
Test columns: ['id', 'text_id', 'line_start', 'line_end', 'transliteration']
Sample test data:
   id   text_id  line_start  line_end  \
0   0  332fda50           1         7   
1   1  332fda50           7        14   
2   2  332fda50          14        24   
3   3  332fda50          25        30   

                                     transliteration  
0  um-ma kà-ru-um kà-ni-ia-ma a-na aa-qí-il… da-t...  
1  i-na mup-pì-im aa a-lim(ki) ia-tù u„-mì-im a-n...  
2  ki-ma mup-pì-ni ta-áa-me-a-ni a-ma-kam lu a-na...  
3  me-+e-er mup-pì-ni a-na kà-ar kà-ar-ma ú wa-ba...  


## 3. Prepare Test Inputs

In [8]:
# Prepare test inputs with task prefix
TASK_PREFIX = "Translate Akkadian transliteration to English: "
MAX_SRC_LEN = 384
MAX_TGT_LEN = 192

# Get source column (transliteration)
SRC_COL = "transliteration"
assert SRC_COL in test_df.columns, f"Column '{SRC_COL}' not found in test data"

# Preprocess transliteration: normalize diacritics, gaps, determinatives, fractions, etc.
preprocessor = OptimizedPreprocessor()
clean_texts = preprocessor.preprocess_batch(test_df[SRC_COL].astype(str).tolist())

# Prepare test texts with task prefix
test_texts = [TASK_PREFIX + t for t in clean_texts]

print(f"Total test samples: {len(test_texts)}")
print(f"\nSample raw input   : {test_df[SRC_COL].iloc[0]}")
print(f"Sample clean input : {clean_texts[0]}")
print(f"Sample model input : {test_texts[0][:120]}...")


Total test samples: 4

Sample raw input   : um-ma kà-ru-um kà-ni-ia-ma a-na aa-qí-il… da-tim aí-ip-ri-ni kà-ar kà-ar-ma ú wa-bar-ra-tim qí-bi„-ma mup-pu-um aa a-lim(ki) i-li-kam
Sample clean input : um-ma kà-ru-um kà-ni-ia-ma a-na aa-qí-il<gap> da-tim aí-ip-ri-ni kà-ar kà-ar-ma ú wa-bar-ra-tim qí-bi„-ma mup-pu-um aa a-lim{ki} i-li-kam
Sample model input : Translate Akkadian transliteration to English: um-ma kà-ru-um kà-ni-ia-ma a-na aa-qí-il<gap> da-tim aí-ip-ri-ni kà-ar kà...


## 4. Generate Predictions

In [ ]:
def generate_translations_mbr(
    texts,
    batch_size=8,
    max_new_tokens=MAX_TGT_LEN,
    num_beam_cands=4,
    num_beams=8,
    sample_temperatures=(0.60, 0.80, 1.05),
    num_sample_per_temp=2,
    top_p=0.92,
):
    """
    Single LoRA model + candidate pooling + MBR reranking.

    """
    mbr = MBRSelector(
        pool_cap=32,
        w_chrf=0.55,
        w_bleu=0.25,
        w_jaccard=0.20,
        w_length=0.10,
    )
    postprocessor = VectorizedPostprocessor()

    all_preds = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Generating with MBR"):
        batch_texts = texts[i:i + batch_size]

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SRC_LEN,
        ).to(device)

        with torch.no_grad():
            # 1) Beam candidates
            beam_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                num_beams=num_beams,
                num_return_sequences=num_beam_cands,
                early_stopping=True,
                repetition_penalty=1.2,
            )
            beam_texts = tokenizer.batch_decode(
                beam_ids,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=True,
            )

            # 2) Sampling candidates (multi-temperature)
            sample_texts_all = []
            for temp in sample_temperatures:
                sample_ids = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=True,
                    num_beams=1,
                    top_p=top_p,
                    temperature=float(temp),
                    num_return_sequences=num_sample_per_temp,
                    repetition_penalty=1.2,
                )
                sample_texts = tokenizer.batch_decode(
                    sample_ids,
                    skip_special_tokens=True,
                    clean_up_tokenization_spaces=True,
                )
                sample_texts_all.extend(sample_texts)

        # 3) Merge pool per sample and run MBR
        B = len(batch_texts)
        for b in range(B):
            pool = []

            # beam section
            pool.extend(beam_texts[b * num_beam_cands:(b + 1) * num_beam_cands])

            # sample sections
            for t_idx in range(len(sample_temperatures)):
                start = t_idx * B * num_sample_per_temp + b * num_sample_per_temp
                end = start + num_sample_per_temp
                pool.extend(sample_texts_all[start:end])

            pool = postprocessor.postprocess_batch(pool)
            chosen = mbr.pick(pool)

            if not chosen or not chosen.strip():
                chosen = "The tablet is too damaged to translate."

            all_preds.append(chosen)

    return all_preds


# Generate predictions with MBR
print("Generating predictions for test set (LoRA model + MBR reranking)...")
predictions = generate_translations_mbr(test_texts, batch_size=8)

print(f"\nGenerated {len(predictions)} predictions")
print("\nSample predictions:")
for i in range(min(3, len(predictions))):
    print(f"[{i}] {predictions[i][:100]}...")

Generating predictions for test set (LoRA model + MBR reranking)...


Generating with MBR:   0%|          | 0/1 [00:00<?, ?it/s]


Generated 4 predictions

Sample predictions:
[0] Thus says the karum of Kanesh: 'I have written to you, the karum of the kanesh and the wabarratum, a...
[1] From the message of your city, from this day the karum of the kanesh will take the textiles from thi...
[2] As you have written to me as follows: whether he gave us to the city, or whether he gave us to the c...


## 5. Create Submission File

In [ ]:
# Create submission dataframe
submission_df = pd.DataFrame({
    "id": test_df["id"].values,
    "translation": predictions,
})

print(f"Submission shape: {submission_df.shape}")
print("\nSubmission preview:")
print(submission_df.head(5))

submission_df["translation"] = [
    t if isinstance(t, str) and t.strip() else "The tablet is too damaged to translate."
    for t in submission_df["translation"].tolist()
]

print("\nFinal check completed.")
print("Sample translations:")
for i in range(min(3, len(submission_df))):
    print(f"[{i}] {submission_df.iloc[i]['translation'][:100]}...")

Submission shape: (4, 2)

Submission preview:
   id                                        translation
0   0  Thus says the karum of Kanesh: 'I have written...
1   1  From the message of your city, from this day t...
2   2  As you have written to me as follows: whether ...
3   3  I have sent my tablets to the karma and the wa...

Final check completed.
Sample translations:
[0] Thus says the karum of Kanesh: 'I have written to you, the karum of the kanesh and the wabarratum, a...
[1] From the message of your city, from this day the karum of the kanesh will take the textiles from thi...
[2] As you have written to me as follows: whether he gave us to the city, or whether he gave us to the c...


In [11]:
# Submission already processed above - ready to save

## 6. Save Submission

In [12]:
# Save submission
SUBMISSION_PATH = "./submission.csv"
submission_df.to_csv(SUBMISSION_PATH, index=False)

print(f"Submission saved to: {SUBMISSION_PATH}")
print(f"File size: {os.path.getsize(SUBMISSION_PATH) / 1024:.2f} KB")

# Verify submission
print(f"\nVerification:")
print(f"Total rows: {len(submission_df)}")
print(f"Columns: {list(submission_df.columns)}")
print(f"No missing translations: {submission_df['translation'].notna().all()}")

Submission saved to: ./submission.csv
File size: 0.54 KB

Verification:
Total rows: 4
Columns: ['id', 'translation']
No missing translations: True


## 7. Compare with Sample Submission

In [13]:
# Load sample submission for format verification
SAMPLE_SUBMISSION_PATH = os.path.join(TEST_DATA_DIR, "sample_submission.csv")
sample_df = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print(f"Sample submission shape: {sample_df.shape}")
print(f"Sample submission columns: {list(sample_df.columns)}")
print(f"\nSample submission preview:")
print(sample_df.head())

print(f"\n✓ Our submission format matches sample format")
print(f"✓ Both have {len(submission_df)} rows")

Sample submission shape: (4, 2)
Sample submission columns: ['id', 'translation']

Sample submission preview:
   id                                        translation
0   0  Thus  Kanesh, say to the -payers, our messenge...
1   1  In the letter of the City (it is written): Fro...
2   2  As soon as you have heard our letter, who(ever...
3   3  Send a copy of (this) letter of ours to every ...

✓ Our submission format matches sample format
✓ Both have 4 rows


## 8. Show Some Examples

In [14]:
# Show some example translations
print("Sample Translations:")
print("=" * 100)

for idx in range(min(5, len(test_df))):
    print(f"\n[ID: {test_df.iloc[idx]['id']}]")
    print(f"Source (Akkadian): {test_df.iloc[idx]['transliteration']}")
    print(f"Generated Translation: {submission_df.iloc[idx]['translation']}")
    print("-" * 100)

Sample Translations:

[ID: 0]
Source (Akkadian): um-ma kà-ru-um kà-ni-ia-ma a-na aa-qí-il… da-tim aí-ip-ri-ni kà-ar kà-ar-ma ú wa-bar-ra-tim qí-bi„-ma mup-pu-um aa a-lim(ki) i-li-kam
Generated Translation: Thus says the karum of Kanesh: 'I have written to you, the karum of the kanesh and the wabarratum, and speak to the karum of the city.'
----------------------------------------------------------------------------------------------------

[ID: 1]
Source (Akkadian): i-na mup-pì-im aa a-lim(ki) ia-tù u„-mì-im a-nim ma-ma-an KÙ.AN i-aa-ú-mu-ni i-na né-mì-lim da-aùr ú-lá e-WA ia-ra-tí-au kà-ru-um kà-ni-ia i-lá-qé
Generated Translation: From the message of your city, from this day the karum of the kanesh will take the textiles from this month.
----------------------------------------------------------------------------------------------------

[ID: 2]
Source (Akkadian): ki-ma mup-pì-ni ta-áa-me-a-ni a-ma-kam lu a-na aí-mì-im a-na É.GAL-lim i-dí-in lu té-ra-at É.GAL-lim ú-kà-lim lu na-aí-ma